# UAT Agent — user perspective

**What it does.** Validates a User Story end-to-end against its acceptance criteria. Produces a `UATResult` with one `Scenario` per `[AC-N]`.

**Entry point.** `await run_uat_and_validate(user_story_id, acceptance_criteria, uat_cycle)` — async coroutine.

**Capability boundary (UATA-04).** `allowed_tools=["Read","Glob","Grep","Bash"]`, `mcp_servers=None`. UAT cannot write to Linear — the Global Orchestrator persists results post-validation.

**Scope discipline.** Every prompt prepends a `SCOPE BOUNDARY:` block listing `[AC-1]`, `[AC-2]`, ... — findings that don't reference one of those literals are out of scope and rejected by the G10 banned-token regex on the orchestrator side.

**Cost.** Live runs consume tokens. Gated on `HSB_NOTEBOOK_RUN_LIVE=1`.

In [ ]:
import sys
from pathlib import Path

# Robust _helpers.py discovery — works whether jupyter lab was launched from
# the repo root, notebooks/, or notebooks/agents/. The loop walks up from cwd
# until it finds the parent that holds _helpers.py and inserts it on sys.path.
nb_dir = Path.cwd()
for _parent in (nb_dir, *nb_dir.parents):
    if (_parent / "_helpers.py").exists():
        sys.path.insert(0, str(_parent))
        break

from _helpers import ensure_src_on_path, runtime_summary

ROOT = ensure_src_on_path()
print("HSB_RUNTIME_<AGENT> selection:\n" + runtime_summary())

## Construct the inputs

UAT takes plain function arguments (no Pydantic input model). The acceptance criteria list becomes the `[AC-N]` scope block in the prompt — order matters.

In [ ]:
example_user_story_id = "LIN-US-7"
example_acceptance_criteria = [
    "README.md exists at the repo root",
    "README.md mentions the project name in the first 5 lines",
]
example_uat_cycle = 1  # 1-indexed; 1=first run, 2=second, 3=cap
print("user_story_id:", example_user_story_id)
print("ACs:")
for i, ac in enumerate(example_acceptance_criteria, 1):
    print(f"  [AC-{i}] {ac}")

## Live invocation

In [ ]:
from _helpers import assert_g1_safe, gated, live_mode

if not live_mode():
    print(gated("UAT live run"))
else:
    assert_g1_safe()
    import asyncio

    from hsb.agents.uat_agent import run_uat_and_validate

    out = asyncio.run(
        run_uat_and_validate(
            user_story_id=example_user_story_id,
            acceptance_criteria=example_acceptance_criteria,
            uat_cycle=example_uat_cycle,
        )
    )
    print("status:", out.overall_status)
    print("scenarios:", len(out.scenarios))
    for s in out.scenarios:
        print(f"  {s.criterion_id}: {s.status}")